# 03 — Benchmark MIDAS vs pyFAI

Head-to-head comparison on the same calibrant frame. Backs Paper A's Table 4.

Install pyFAI via the `[paper]` extra: `pip install 'midas-auto-calibrate[paper]'`.

In [1]:
import shutil, tempfile
from pathlib import Path

import midas_auto_calibrate as mac

try:
    import pyFAI
    print('pyFAI', pyFAI.version)
except ImportError:
    print('pyFAI not installed — the benchmark will still run MIDAS but skip pyFAI.')

pyFAI 2026.2.1


## Stage data + invoke `benchmark()`

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mac_bench_'))
img = workdir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img)
shutil.copy(mac.data.CEO2_PILATUS_DARK, workdir / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, workdir / 'mask_upd.tif')

result = mac.benchmark(
    img,
    material='CeO2',
    wavelength=0.172973,
    pixel_size=172.0,
    nr_pixels_y=1475, nr_pixels_z=1679,
    lsd=657_436.9, ybc=685.5, zbc=921.0,
    n_iterations=5,
    work_dir=workdir,
    include_pyfai=True,
)

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/asyncio/base_events.py", line 639, in run_forever
    self._run_once()
  File "/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/asyncio/base_events.py", line 1985, in _run_once
    handle._run()
  File "

/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/midas_auto_calibrate/benchmark.py:311: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(


## Tabulate

`as_dict()` gives the headline numbers the paper reports.

In [3]:
for k, v in result.as_dict().items():
    print(f'  {k:>16}: {v:.3f}')

     midas_ustrain: 2151.368
     midas_seconds: 7.290
     pyfai_ustrain: 3930.879
     pyfai_seconds: 8.463
    accuracy_ratio: 1.827
           speedup: 1.161


## Per-ring pyFAI strain vs MIDAS single-value strain

MIDAS reports a single mean pseudo-strain across all rings; pyFAI (as used
here) computes per-ring strain from 2D integration + pseudo-Voigt fits.

In [4]:
if result.pyfai_per_ring_ustrain:
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 4))
        rings = range(1, len(result.pyfai_per_ring_ustrain) + 1)
        ax.bar(rings, result.pyfai_per_ring_ustrain, alpha=0.7, label='pyFAI per ring')
        ax.axhline(result.midas_pseudo_strain, color='tab:red', linestyle='--',
                   label=f'MIDAS mean = {result.midas_pseudo_strain:.1f} µε')
        ax.set_xlabel('Ring #')
        ax.set_ylabel('Pseudo-strain (µε)')
        ax.set_title(f'{result.material} @ {result.wavelength} Å')
        ax.legend()
        fig.tight_layout()
        plt.show()
    except ImportError:
        print('matplotlib not installed')
else:
    print('pyFAI not installed or no rings fit — matplotlib plot skipped.')

/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/ipykernel_48947/4090468093.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
